### Analyse RAG Results from Florent (12 June 2026)

In [1]:
#!pip install wordcloud 

In [1]:
"""
Generate benchmark visualisation figures from benchmark_results.csv.

Figure 1: Score label distribution per question (stacked bar).
Figure 2: Score label distribution per document (stacked bar).

Run from the project root:
  pixi run plot-benchmark
"""

import json
import os
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [ ]:
from pathlib import Path
from config_loader import load_config

cfg              = load_config()
ANALYSIS_DIR     = cfg['paths']['analysis_dir']
BENCHMARK_OUTPUT = Path(cfg['paths']['benchmark_output_dir'])
PLOT_OUTPUT_DIR  = Path(cfg['paths']['fig_folder'])

print('BENCHMARK_OUTPUT:', BENCHMARK_OUTPUT)
print('PLOT_OUTPUT_DIR :', PLOT_OUTPUT_DIR)


In [11]:
SCORE_ORDER = ["Correct", "Partially Correct", "Undefined", "Missing", "Wrong"]

COLORS = {
    "Correct":           "#2166AC",
    "Partially Correct": "#92C5DE",
    "Undefined":         "#D1D1D1",
    "Missing":           "#F4A582",
    "Wrong":             "#B2182B",
}

QUESTION_SHORT = {
    9:  "[Q1]:  Scenarios listed",
    10: "[Q2]: Temperature",
    11: "[Q3]: Scenario provider",
    12: "[Q4]: Pick-and-mix SSP/RCP",
    13: "[Q5]: Customised scenarios",
    14: "[Q6]: Co-produced scenarios",
    15: "[Q7] Baseline scenario",
    16: "[Q8]: Models used",
    17: "[Q9]: Acute physical risk damage",
    18: "[Q10]: Non-temperature variables",
    19: "[Q11]: Scale of scenarios",
    20: "[Q12]: Time frames",
    21: "[Q13]: Justification for choice",
}


def shorten_company(name: str) -> str:
    return (
        name.replace("_LIMITED", "")
            .replace("_GROUP", "")
            .replace("_NEW_ZEALAND", "_NZ")
            .replace("_INTERNATIONAL", "_INT")
            .replace("_SERVICES", "")
            .replace("_INFRASTRUCTURE", "_INFRA")
            .replace("_INVESTMENTS", "_INV")
    )


def stacked_bar(ax, counts_df: pd.DataFrame, title: str, ylabel: str):
    y = np.arange(len(counts_df))
    left = np.zeros(len(counts_df))

    for label in SCORE_ORDER:
        if label not in counts_df.columns:
            continue
        values = counts_df[label].values.astype(float)
        ax.barh(y, values, left=left, color=COLORS[label], height=0.6)
        left += values

    ax.set_title(title, fontsize=12, fontweight="bold", pad=10)
    ax.set_xlabel("Number of evaluations", fontsize=10)
    #ax.set_ylabel(ylabel, fontsize=10)
    ax.set_yticks(y)
    ax.set_yticklabels(counts_df.index, fontsize=8)
    ax.invert_yaxis()
    ax.xaxis.grid(True, linestyle="--", alpha=0.5)
    ax.set_axisbelow(True)


SCORE_LABELS = {
    1: "Wrong",
    2: "Undefined",
    3: "Missing",
    4: "Partially Correct",
    5: "Correct",
}


def load_results() -> pd.DataFrame:
    rows = []
    for path in sorted(BENCHMARK_OUTPUT.glob("*.json")):
        data = json.loads(path.read_text())
        company = data.get("source", path.stem)
        for answer in data.get("answers", []):
            raw = answer.get("verification")
            if raw is None:
                continue
            score = max(1, min(5, int(raw)))
            rows.append({
                "company":    company,
                "q_id":       int(answer["q_id"]),
                "score":      score,
                "score_label": SCORE_LABELS[score],
            })
    return pd.DataFrame(rows)


In [ ]:
df = load_results()
PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Company to Industry Mapping (fill in the industry for each company)
COMPANY_TO_INDUSTRY = {
    'AIA_NEW_ZEALAND_LIMITED_2023': 'Finance - Insurance',
    'BRISCOEE_GROUP_LIMITED_2024': 'Other industries',
    'CDL_INVESTMENTS_NEW_ZEALAND_LIMITED_2023': 'Finance - Other',
    'CHANNEL_INFRASTRUCTURE_NZ_LIMITED_2023': 'Infrastructure',
    'CHUBB_LIFE_INSURANCE_NEW_ZEALAND_LIMITED_2023': 'Finance - Insurance',
    'CITIBANK_NA_NEW_ZEALAND_BRANCH_2023': 'Finance - Banks',
    'MILLENNIUM_AND_COPTHORNE_HOTELS_NEW_ZEALAND_LIMITED_2023': 'Other industries',
    'NZME_LIMITED_2023': 'Other industries',
    'PFI_LIMITED_2023': 'Real estates',
    'RABOBANK_NEW_ZEALAND_LIMITED_2023':'Finance - Banks',
    'VENTIA_SERVICES_GROUP_LIMITED_2023': 'Other industries',
    'VISTA_GROUP_INTERNATIONAL_LIMITED_2023': 'Telecom',
    'Z_ENERGY_LIMITED_2023': 'Energy',
}
# Build anonymised mapping per industry (numbered)
unique_comps = pd.Series(df['company'].unique())
mapping = {}
counters = {}
for comp in unique_comps:
    ind = COMPANY_TO_INDUSTRY.get(comp, 'Unknown')
    counters.setdefault(ind, 0)
    counters[ind] += 1
    mapping[comp] = f"{ind} company {counters[ind]}"

# Add anonymised column
df['company_anonym'] = df['company'].map(mapping)

legend_patches = [
    mpatches.Patch(color=COLORS[l], label=l) for l in SCORE_ORDER
]

q_counts = (
    df.groupby(["q_id", "score_label"]) 
    .size()
    .unstack(fill_value=0)
    .reindex(columns=SCORE_ORDER, fill_value=0)
)
q_counts.index = [QUESTION_SHORT.get(i, f"Q{i}") for i in q_counts.index]

# Group by anonymised company labels for document-level counts
doc_counts = (
    df.groupby(["company_anonym", "score_label"]) 
    .size()
    .unstack(fill_value=0)
    .reindex(columns=SCORE_ORDER, fill_value=0)
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 10))

stacked_bar(ax1, q_counts, "Score distribution per question", "Question")
stacked_bar(ax2, doc_counts, "Score distribution per document", "Document")

ax1.text(-0.01, 1.05, "a)", transform=ax1.transAxes, fontsize=12,
            fontweight="bold", va="bottom", ha="right")
ax2.text(-0.01, 1.05, "b)", transform=ax2.transAxes, fontsize=12,
            fontweight="bold", va="bottom", ha="right")

ax2.legend(handles=legend_patches, title="Score", bbox_to_anchor=(1.01, 1),
            loc="upper left", fontsize=9, title_fontsize=9)

fig.tight_layout()
out = PLOT_OUTPUT_DIR / "Fig09_benchmark_scores.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved {out}")
plt.show()

NameError: name 'detect_industry' is not defined

In [9]:
df

""


In [ ]:

# ── Summary table ─────────────────────────────────────────────────────────
total = len(df)
print(f"\nOverall ({total} evaluations):")
for label in SCORE_ORDER:
    n = (df["score_label"] == label).sum()
    print(f"  {label:<20} {n:>3}  ({100*n/total:.0f}%)")